In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_IGI_Airport_(T3)_Delhi_IMD_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,347.0,149.0,198.0,87.0,158.0,172.0,NaN,23.0,NaN,141.0,361.0,280.0
1,2,302.0,NaN,151.0,117.0,NaN,98.0,79.0,NaN,NaN,159.0,339.0,273.0
2,3,349.0,154.0,NaN,108.0,271.0,107.0,NaN,57.0,120.0,144.0,368.0,235.0
3,4,344.0,256.0,122.0,112.0,290.0,139.0,47.0,68.0,63.0,195.0,381.0,165.0
4,5,NaN,137.0,NaN,136.0,278.0,157.0,75.0,NaN,74.0,129.0,368.0,139.0
5,6,NaN,129.0,NaN,113.0,218.0,157.0,NaN,48.0,126.0,135.0,347.0,138.0
6,7,270.0,156.0,134.0,146.0,293.0,226.0,NaN,NaN,116.0,110.0,367.0,226.0
7,8,318.0,NaN,122.0,136.0,170.0,183.0,NaN,49.0,66.0,135.0,NaN,277.0
8,9,331.0,111.0,NaN,160.0,137.0,124.0,NaN,82.0,84.0,150.0,342.0,173.0
9,10,244.0,319.0,145.0,NaN,135.0,104.0,NaN,57.0,128.0,125.0,324.0,218.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,347.000000,149.000000,125.548387,87.00000,158.000000,172.000000,68.869565,60.0,91.366667,141.00000,361.000000,280.000000
1,2,302.000000,173.129032,151.000000,117.00000,156.333333,98.000000,79.000000,60.0,91.366667,159.00000,339.000000,273.000000
2,3,349.000000,154.000000,125.548387,108.00000,156.333333,107.000000,68.869565,57.0,120.000000,144.00000,368.000000,235.000000
3,4,344.000000,256.000000,122.000000,112.00000,156.333333,139.000000,68.869565,68.0,63.000000,195.00000,381.000000,165.000000
4,5,285.741935,137.000000,125.548387,136.00000,156.333333,157.000000,75.000000,60.0,74.000000,129.00000,368.000000,139.000000
5,6,285.741935,129.000000,125.548387,113.00000,218.000000,157.000000,68.869565,48.0,126.000000,135.00000,347.000000,138.000000
6,7,270.000000,156.000000,134.000000,146.00000,156.333333,107.323529,68.869565,60.0,116.000000,110.00000,367.000000,226.000000
7,8,318.000000,173.129032,122.000000,136.00000,170.000000,183.000000,68.869565,49.0,66.000000,135.00000,300.966667,277.000000
8,9,331.000000,111.000000,125.548387,160.00000,137.000000,124.000000,68.869565,82.0,84.000000,150.00000,342.000000,173.000000
9,10,244.000000,173.129032,145.000000,135.53125,135.000000,104.000000,68.869565,57.0,128.000000,125.00000,324.000000,218.000000
